# Sessionisation

## Overview
**Sessionisation** groups a sequence of user events into discrete sessions -- bursts of activity separated by periods of inactivity. It is the foundation of web analytics, app engagement metrics, and behavioural segmentation.

**The gap-based approach:**
1. Order each user's events by time
2. Compute the gap between each event and the previous one using `LAG`
3. Mark a new session whenever the gap exceeds a threshold (e.g. 30 minutes or 1 day)
4. Assign session IDs using a cumulative sum of the new-session flags

```
event_time          gap_minutes   new_session   session_id
2023-01-10 09:00    NULL          1             1
2023-01-10 09:05    5             0             1
2023-01-10 09:12    7             0             1
2023-01-10 14:00    288           1             2   <- gap > 60 min
2023-01-10 14:03    3             0             2
```

**Session gap threshold choice:**
- Web analytics: 30 minutes (industry standard)
- Mobile app: 5-15 minutes depending on use case
- Banking transactions: 1 day (each day's activity = one session)
- The right threshold depends on natural usage patterns in your product

**Key session metrics:** sessions per user per period, session length (first to last event), events per session, bounce rate (sessions with only 1 event).

**PostgreSQL notes:** `EXTRACT(EPOCH FROM interval)` gives gap in seconds. In SQLite, use `(JULIANDAY(t2) - JULIANDAY(t1)) * 86400` for seconds or `* 1440` for minutes.

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, segment TEXT, province TEXT, signup_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT, opened_date TEXT, status TEXT, credit_limit REAL);CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, customer_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT);CREATE TABLE product_events (event_id INTEGER PRIMARY KEY, customer_id INTEGER, event_date TEXT, event_type TEXT, product TEXT, channel TEXT);INSERT INTO customers VALUES (1,'Aroha','Ngata','Retail','NB','2023-01-05',1),(2,'Liam','Chen','Wealth','NS','2023-01-12',1),(3,'Fatima','Al-Rashid','SME','ON','2023-01-20',1),(4,'James','MacLeod','Retail','NB','2023-02-03',1),(5,'Sofia','Petrov','Retail','BC','2023-02-14',1),(6,'Noah','Williams','SME','AB','2023-02-28',0),(7,'Mei','Zhang','Wealth','ON','2023-03-10',1),(8,'Ethan','Tremblay','Retail','QC','2023-04-05',1),(9,'Priya','Nair','Wealth','BC','2023-04-18',1),(10,'Marcus','Okafor','SME','ON','2023-04-25',1),(11,'Diana','Flores','Retail','NB','2023-05-02',1),(12,'Ravi','Patel','Retail','ON','2023-05-15',1),(13,'Yuki','Tanaka','Wealth','BC','2023-06-01',1),(14,'Omar','Hassan','SME','QC','2023-06-20',1),(15,'Chloe','Bouchard','Retail','QC','2023-07-08',0);INSERT INTO accounts VALUES (101,1,'Chequing','2023-01-05','Active',NULL),(102,1,'Savings','2023-01-05','Active',NULL),(103,2,'Investment','2023-01-12','Active',50000),(104,3,'Chequing','2023-01-20','Active',NULL),(105,3,'Loan','2023-01-20','Active',25000),(106,4,'Chequing','2023-02-03','Active',NULL),(107,5,'Chequing','2023-02-14','Active',NULL),(108,5,'Savings','2023-02-14','Active',NULL),(109,6,'Chequing','2023-02-28','Closed',NULL),(110,7,'Investment','2023-03-10','Active',75000),(111,8,'Chequing','2023-04-05','Active',NULL),(112,9,'Investment','2023-04-18','Active',100000),(113,10,'Chequing','2023-04-25','Active',NULL),(114,10,'Loan','2023-04-25','Active',15000),(115,11,'Chequing','2023-05-02','Active',NULL),(116,12,'Chequing','2023-05-15','Active',NULL),(117,12,'Savings','2023-05-15','Active',NULL),(118,13,'Investment','2023-06-01','Active',60000),(119,14,'Chequing','2023-06-20','Active',NULL),(120,15,'Chequing','2023-07-08','Closed',NULL);INSERT INTO transactions VALUES (1001,101,1,'2023-01-10','Deposit',2500.00,'Payroll'),(1002,101,1,'2023-01-22','Withdrawal',-800.00,'Rent'),(1003,102,1,'2023-02-05','Deposit',500.00,'Transfer'),(1004,103,2,'2023-02-10','Deposit',10000.00,'Investment'),(1005,104,3,'2023-02-15','Deposit',3200.00,'Payroll'),(1006,104,3,'2023-03-01','Withdrawal',-1200.00,'Rent'),(1007,106,4,'2023-03-10','Deposit',2800.00,'Payroll'),(1008,107,5,'2023-03-15','Deposit',2200.00,'Payroll'),(1009,107,5,'2023-03-28','Fee',-25.00,'Monthly Fee'),(1010,101,1,'2023-03-10','Deposit',2500.00,'Payroll'),(1011,103,2,'2023-04-15','Deposit',15000.00,'Investment'),(1012,110,7,'2023-04-20','Deposit',20000.00,'Investment'),(1013,111,8,'2023-05-01','Deposit',2100.00,'Payroll'),(1014,112,9,'2023-05-10','Deposit',25000.00,'Investment'),(1015,113,10,'2023-05-20','Deposit',3500.00,'Payroll'),(1016,115,11,'2023-06-01','Deposit',2000.00,'Payroll'),(1017,116,12,'2023-06-15','Deposit',2700.00,'Payroll'),(1018,118,13,'2023-07-01','Deposit',18000.00,'Investment'),(1019,101,1,'2023-07-10','Deposit',2500.00,'Payroll'),(1020,103,2,'2023-07-20','Deposit',12000.00,'Investment'),(1021,104,3,'2023-07-15','Deposit',3200.00,'Payroll'),(1022,107,5,'2023-08-01','Deposit',2200.00,'Payroll'),(1023,111,8,'2023-08-05','Deposit',2100.00,'Payroll'),(1024,113,10,'2023-08-15','Withdrawal',-500.00,'Transfer'),(1025,116,12,'2023-08-20','Deposit',2700.00,'Payroll'),(1026,101,1,'2023-10-10','Deposit',2500.00,'Payroll'),(1027,116,12,'2023-10-20','Deposit',2700.00,'Payroll');INSERT INTO product_events VALUES (1,1,'2023-01-10','page_view','Savings Account','web'),(2,1,'2023-01-10','product_view','Savings Account','web'),(3,1,'2023-01-10','application_start','Savings Account','web'),(4,1,'2023-01-11','application_submit','Savings Account','web'),(5,1,'2023-01-15','account_opened','Savings Account','web'),(6,2,'2023-01-12','page_view','Investment Account','mobile'),(7,2,'2023-01-12','product_view','Investment Account','mobile'),(8,2,'2023-01-12','application_start','Investment Account','mobile'),(9,2,'2023-01-13','application_submit','Investment Account','mobile'),(10,2,'2023-01-15','account_opened','Investment Account','mobile'),(11,3,'2023-01-20','page_view','Chequing Account','web'),(12,3,'2023-01-20','product_view','Chequing Account','web'),(13,3,'2023-01-20','application_start','Chequing Account','web'),(14,4,'2023-02-03','page_view','Chequing Account','web'),(15,4,'2023-02-03','product_view','Chequing Account','web'),(16,5,'2023-02-14','page_view','Savings Account','mobile'),(17,5,'2023-02-14','product_view','Savings Account','mobile'),(18,5,'2023-02-14','application_start','Savings Account','mobile'),(19,5,'2023-02-15','application_submit','Savings Account','mobile'),(20,5,'2023-02-20','account_opened','Savings Account','mobile'),(21,7,'2023-03-10','page_view','Investment Account','web'),(22,7,'2023-03-10','product_view','Investment Account','web'),(23,7,'2023-03-10','application_start','Investment Account','web'),(24,7,'2023-03-11','application_submit','Investment Account','web'),(25,7,'2023-03-15','account_opened','Investment Account','web'),(26,8,'2023-04-05','page_view','Chequing Account','mobile'),(27,8,'2023-04-05','product_view','Chequing Account','mobile'),(28,9,'2023-04-18','page_view','Investment Account','web'),(29,9,'2023-04-18','product_view','Investment Account','web'),(30,9,'2023-04-18','application_start','Investment Account','web'),(31,9,'2023-04-20','application_submit','Investment Account','web'),(32,9,'2023-04-25','account_opened','Investment Account','web');""")
conn.commit()
print("Finance schema ready.")
for t in ["customers","accounts","transactions","product_events"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

Finance schema ready.
  customers: 15 rows
  accounts: 20 rows
  transactions: 27 rows
  product_events: 32 rows


---
## Gap detection with LAG

In [2]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, segment TEXT, province TEXT, signup_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT, opened_date TEXT, status TEXT, credit_limit REAL);CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, customer_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT);CREATE TABLE product_events (event_id INTEGER PRIMARY KEY, customer_id INTEGER, event_date TEXT, event_type TEXT, product TEXT, channel TEXT);INSERT INTO customers VALUES (1,'Aroha','Ngata','Retail','NB','2023-01-05',1),(2,'Liam','Chen','Wealth','NS','2023-01-12',1),(3,'Fatima','Al-Rashid','SME','ON','2023-01-20',1),(4,'James','MacLeod','Retail','NB','2023-02-03',1),(5,'Sofia','Petrov','Retail','BC','2023-02-14',1),(6,'Noah','Williams','SME','AB','2023-02-28',0),(7,'Mei','Zhang','Wealth','ON','2023-03-10',1),(8,'Ethan','Tremblay','Retail','QC','2023-04-05',1),(9,'Priya','Nair','Wealth','BC','2023-04-18',1),(10,'Marcus','Okafor','SME','ON','2023-04-25',1),(11,'Diana','Flores','Retail','NB','2023-05-02',1),(12,'Ravi','Patel','Retail','ON','2023-05-15',1),(13,'Yuki','Tanaka','Wealth','BC','2023-06-01',1),(14,'Omar','Hassan','SME','QC','2023-06-20',1),(15,'Chloe','Bouchard','Retail','QC','2023-07-08',0);INSERT INTO accounts VALUES (101,1,'Chequing','2023-01-05','Active',NULL),(102,1,'Savings','2023-01-05','Active',NULL),(103,2,'Investment','2023-01-12','Active',50000),(104,3,'Chequing','2023-01-20','Active',NULL),(105,3,'Loan','2023-01-20','Active',25000),(106,4,'Chequing','2023-02-03','Active',NULL),(107,5,'Chequing','2023-02-14','Active',NULL),(108,5,'Savings','2023-02-14','Active',NULL),(109,6,'Chequing','2023-02-28','Closed',NULL),(110,7,'Investment','2023-03-10','Active',75000),(111,8,'Chequing','2023-04-05','Active',NULL),(112,9,'Investment','2023-04-18','Active',100000),(113,10,'Chequing','2023-04-25','Active',NULL),(114,10,'Loan','2023-04-25','Active',15000),(115,11,'Chequing','2023-05-02','Active',NULL),(116,12,'Chequing','2023-05-15','Active',NULL),(117,12,'Savings','2023-05-15','Active',NULL),(118,13,'Investment','2023-06-01','Active',60000),(119,14,'Chequing','2023-06-20','Active',NULL),(120,15,'Chequing','2023-07-08','Closed',NULL);INSERT INTO transactions VALUES (1001,101,1,'2023-01-10','Deposit',2500.00,'Payroll'),(1002,101,1,'2023-01-22','Withdrawal',-800.00,'Rent'),(1003,102,1,'2023-02-05','Deposit',500.00,'Transfer'),(1004,103,2,'2023-02-10','Deposit',10000.00,'Investment'),(1005,104,3,'2023-02-15','Deposit',3200.00,'Payroll'),(1006,104,3,'2023-03-01','Withdrawal',-1200.00,'Rent'),(1007,106,4,'2023-03-10','Deposit',2800.00,'Payroll'),(1008,107,5,'2023-03-15','Deposit',2200.00,'Payroll'),(1009,107,5,'2023-03-28','Fee',-25.00,'Monthly Fee'),(1010,101,1,'2023-03-10','Deposit',2500.00,'Payroll'),(1011,103,2,'2023-04-15','Deposit',15000.00,'Investment'),(1012,110,7,'2023-04-20','Deposit',20000.00,'Investment'),(1013,111,8,'2023-05-01','Deposit',2100.00,'Payroll'),(1014,112,9,'2023-05-10','Deposit',25000.00,'Investment'),(1015,113,10,'2023-05-20','Deposit',3500.00,'Payroll'),(1016,115,11,'2023-06-01','Deposit',2000.00,'Payroll'),(1017,116,12,'2023-06-15','Deposit',2700.00,'Payroll'),(1018,118,13,'2023-07-01','Deposit',18000.00,'Investment'),(1019,101,1,'2023-07-10','Deposit',2500.00,'Payroll'),(1020,103,2,'2023-07-20','Deposit',12000.00,'Investment'),(1021,104,3,'2023-07-15','Deposit',3200.00,'Payroll'),(1022,107,5,'2023-08-01','Deposit',2200.00,'Payroll'),(1023,111,8,'2023-08-05','Deposit',2100.00,'Payroll'),(1024,113,10,'2023-08-15','Withdrawal',-500.00,'Transfer'),(1025,116,12,'2023-08-20','Deposit',2700.00,'Payroll'),(1026,101,1,'2023-10-10','Deposit',2500.00,'Payroll'),(1027,116,12,'2023-10-20','Deposit',2700.00,'Payroll');INSERT INTO product_events VALUES (1,1,'2023-01-10','page_view','Savings Account','web'),(2,1,'2023-01-10','product_view','Savings Account','web'),(3,1,'2023-01-10','application_start','Savings Account','web'),(4,1,'2023-01-11','application_submit','Savings Account','web'),(5,1,'2023-01-15','account_opened','Savings Account','web'),(6,2,'2023-01-12','page_view','Investment Account','mobile'),(7,2,'2023-01-12','product_view','Investment Account','mobile'),(8,2,'2023-01-12','application_start','Investment Account','mobile'),(9,2,'2023-01-13','application_submit','Investment Account','mobile'),(10,2,'2023-01-15','account_opened','Investment Account','mobile'),(11,3,'2023-01-20','page_view','Chequing Account','web'),(12,3,'2023-01-20','product_view','Chequing Account','web'),(13,3,'2023-01-20','application_start','Chequing Account','web'),(14,4,'2023-02-03','page_view','Chequing Account','web'),(15,4,'2023-02-03','product_view','Chequing Account','web'),(16,5,'2023-02-14','page_view','Savings Account','mobile'),(17,5,'2023-02-14','product_view','Savings Account','mobile'),(18,5,'2023-02-14','application_start','Savings Account','mobile'),(19,5,'2023-02-15','application_submit','Savings Account','mobile'),(20,5,'2023-02-20','account_opened','Savings Account','mobile'),(21,7,'2023-03-10','page_view','Investment Account','web'),(22,7,'2023-03-10','product_view','Investment Account','web'),(23,7,'2023-03-10','application_start','Investment Account','web'),(24,7,'2023-03-11','application_submit','Investment Account','web'),(25,7,'2023-03-15','account_opened','Investment Account','web'),(26,8,'2023-04-05','page_view','Chequing Account','mobile'),(27,8,'2023-04-05','product_view','Chequing Account','mobile'),(28,9,'2023-04-18','page_view','Investment Account','web'),(29,9,'2023-04-18','product_view','Investment Account','web'),(30,9,'2023-04-18','application_start','Investment Account','web'),(31,9,'2023-04-20','application_submit','Investment Account','web'),(32,9,'2023-04-25','account_opened','Investment Account','web');""")
conn.commit()
q = """
SELECT  customer_id, event_date, event_type,
        LAG(event_date) OVER (PARTITION BY customer_id ORDER BY event_date, event_id) AS prev_date,
        CAST(
            (JULIANDAY(event_date) -
             JULIANDAY(LAG(event_date) OVER (PARTITION BY customer_id
                                             ORDER BY event_date, event_id)))
            * 1440  AS INTEGER)  AS gap_minutes
FROM    product_events
WHERE   customer_id = 1
ORDER BY event_date, event_id
"""
print("=== Event gaps for customer 1 ===")
print(pd.read_sql(q, conn).to_string(index=False))
print()
print("NULL gap = first event. Gap > 60 minutes signals a new session.")

=== Event gaps for customer 1 ===
 customer_id event_date         event_type  prev_date  gap_minutes
           1 2023-01-10          page_view       None          NaN
           1 2023-01-10       product_view 2023-01-10          0.0
           1 2023-01-10  application_start 2023-01-10          0.0
           1 2023-01-11 application_submit 2023-01-10       1440.0
           1 2023-01-15     account_opened 2023-01-11       5760.0

NULL gap = first event. Gap > 60 minutes signals a new session.


---
## Assigning session IDs

In [3]:
import pandas as pd
q = """
with gaps AS (
    SELECT  customer_id,
            event_id,
            event_date,
            event_type,
            product,
            channel,
            CASE
                WHEN LAG(event_date) OVER (PARTITION BY customer_id
                                           ORDER BY event_date, event_id) IS NULL
                THEN 1
                WHEN CAST(
                        (JULIANDAY(event_date) -
                         JULIANDAY(LAG(event_date) OVER (
                             PARTITION BY customer_id ORDER BY event_date, event_id)))
                        * 1440 AS INTEGER) > 60
                THEN 1
                ELSE 0
            END  AS new_session_flag
    FROM    product_events
),
sessions AS (
    SELECT  *,
            SUM(new_session_flag) OVER (PARTITION BY customer_id
                                        ORDER BY event_date, event_id
                                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
                AS session_num
    FROM    gaps
)
SELECT  customer_id,
        session_num,
        MIN(event_date)  AS session_start,
        MAX(event_date)  AS session_end,
        COUNT(*)         AS events_in_session,
        GROUP_CONCAT(event_type, ' -> ')  AS event_sequence
FROM    sessions
GROUP BY customer_id, session_num
ORDER BY customer_id, session_num
"""
print("=== Session IDs via cumulative sum of new-session flags ===")
print(pd.read_sql(q, conn).to_string(index=False))

=== Session IDs via cumulative sum of new-session flags ===
 customer_id  session_num session_start session_end  events_in_session                                 event_sequence
           1            1    2023-01-10  2023-01-10                  3 page_view -> product_view -> application_start
           1            2    2023-01-11  2023-01-11                  1                             application_submit
           1            3    2023-01-15  2023-01-15                  1                                 account_opened
           2            1    2023-01-12  2023-01-12                  3 page_view -> product_view -> application_start
           2            2    2023-01-13  2023-01-13                  1                             application_submit
           2            3    2023-01-15  2023-01-15                  1                                 account_opened
           3            1    2023-01-20  2023-01-20                  3 page_view -> product_view -> application_st

---
## Session metrics and bounce detection

In [4]:
import pandas as pd
q = """
with gaps AS (
    SELECT  customer_id, event_id, event_date, event_type, channel,
            CASE
                WHEN LAG(event_date) OVER (PARTITION BY customer_id
                                           ORDER BY event_date, event_id) IS NULL THEN 1
                WHEN CAST((JULIANDAY(event_date) -
                           JULIANDAY(LAG(event_date) OVER (
                               PARTITION BY customer_id ORDER BY event_date, event_id)))
                          * 1440 AS INTEGER) > 60 THEN 1
                ELSE 0
            END  AS new_session_flag
    FROM    product_events
),
sessions AS (
    SELECT  *,
            SUM(new_session_flag) OVER (PARTITION BY customer_id
                                        ORDER BY event_date, event_id
                                        ROWS UNBOUNDED PRECEDING) AS session_num
    FROM    gaps
),
session_summary AS (
    SELECT  customer_id,
            session_num,
            MIN(event_date)           AS session_start,
            MAX(event_date)           AS session_end,
            COUNT(*)                  AS events,
            MAX(channel)              AS channel,
            CAST(
                (JULIANDAY(MAX(event_date)) - JULIANDAY(MIN(event_date)))
                * 1440 AS INTEGER)    AS duration_minutes
    FROM    sessions
    GROUP BY customer_id, session_num
)
SELECT  customer_id,
        COUNT(*)                            AS total_sessions,
        ROUND(AVG(events), 1)               AS avg_events_per_session,
        ROUND(AVG(duration_minutes), 1)     AS avg_duration_min,
        SUM(CASE WHEN events = 1 THEN 1 ELSE 0 END)  AS bounced_sessions,
        ROUND(100.0 * SUM(CASE WHEN events = 1 THEN 1 ELSE 0 END)
              / COUNT(*), 1)                AS bounce_rate_pct
FROM    session_summary
GROUP BY customer_id
ORDER BY customer_id
"""
print("=== Session metrics: length, event count, bounce detection ===")
print(pd.read_sql(q, conn).to_string(index=False))
conn.close()

=== Session metrics: length, event count, bounce detection ===
 customer_id  total_sessions  avg_events_per_session  avg_duration_min  bounced_sessions  bounce_rate_pct
           1               3                     1.7               0.0                 2             66.7
           2               3                     1.7               0.0                 2             66.7
           3               1                     3.0               0.0                 0              0.0
           4               1                     2.0               0.0                 0              0.0
           5               3                     1.7               0.0                 2             66.7
           7               3                     1.7               0.0                 2             66.7
           8               1                     2.0               0.0                 0              0.0
           9               3                     1.7               0.0                 2 

---
## Common Pitfalls

**1. Choosing the gap threshold arbitrarily**
30 minutes is standard for web sessions but wrong for a daily banking app. Analyse the actual inter-event gap distribution first: plot a histogram and look for a natural break point. The threshold should reflect a genuine pause in user intent, not an arbitrary round number.

**2. Not partitioning by user before computing gaps**
`LAG(event_date) OVER (ORDER BY event_date)` without `PARTITION BY customer_id` computes gaps across all users -- the last event of customer 1 bleeds into the first event of customer 2. Always include `PARTITION BY customer_id` in the window.

**3. Using timestamp alone for event ordering when timestamps tie**
Two events with the same timestamp but different event_ids will be ordered arbitrarily without a tiebreaker. Use `ORDER BY event_date, event_id` to ensure stable, reproducible ordering.

**4. Confusing per-user session_num with a globally unique session ID**
`SUM(new_session_flag) OVER (PARTITION BY customer_id ...)` gives session 1, 2, 3 per customer -- not globally unique. Customer 1's session 1 and customer 2's session 1 are different sessions. Construct a unique ID as `customer_id || '-' || session_num` or use `DENSE_RANK()` over all sessions.

**5. Including bounce sessions in engagement averages**
A single `page_view` with no follow-up is a bounce. Including it in "average events per session" or "average session duration" deflates metrics that are meant to measure engaged sessions. Filter `WHERE events > 1` when computing engagement-only metrics.

**6. Event log delays corrupting session boundaries**
Queued or late-arriving events may appear out of chronological order in the log. An event that physically arrives 3 hours late due to a network delay will incorrectly start a new session. Use the client-side event_time rather than server ingestion_time for gap calculations where possible.


---
*sql_methods_library - Samantha McGarrigle*